In [2]:
# Data_Preprocessing.ipynb

# 1. Import Libraries
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 2. Paths
csv_path = '/run/media/bkrishna/New Volume/skin-disease-chatbot/data/HAM10000_metadata.csv'
img_dirs = [
    '/run/media/bkrishna/New Volume/skin-disease-chatbot/data/HAM10000_images_part_1',
    '/run/media/bkrishna/New Volume/skin-disease-chatbot/data/HAM10000_images_part_2'
]

# 3. Load Metadata
df = pd.read_csv(csv_path)
print(f"Metadata loaded: {df.shape}")

# 4. Map Image IDs to Paths
all_img_paths = {}
for img_dir in img_dirs:
    for img_file in os.listdir(img_dir):
        img_id = os.path.splitext(img_file)[0]
        all_img_paths[img_id] = os.path.join(img_dir, img_file)

# 5. Map Labels
label_mapping = {
    'nv': 0,
    'bkl': 1,
    'mel': 2,
    'bcc': 3,
    'akiec': 4,
    'vasc': 5,
    'df': 6
}
df['label'] = df['dx'].map(label_mapping)

# 6. Keep only valid images
valid_df = df[df['image_id'].isin(all_img_paths.keys())]
print(f"Valid dataset: {valid_df.shape}")

# 7. Train-Validation Split
train_df, val_df = train_test_split(valid_df, test_size=0.2, random_state=42, stratify=valid_df['label'])

print(f"Train samples: {train_df.shape}")
print(f"Validation samples: {val_df.shape}")

# 8. Create Custom Data Generator
class SkinDiseaseDataGenerator(Sequence):
    def __init__(self, dataframe, batch_size=32, img_size=(224, 224), shuffle=True):
        self.dataframe = dataframe
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataframe))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataframe) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_df = self.dataframe.iloc[batch_indexes]
        
        images = []
        labels = []

        for _, row in batch_df.iterrows():
            img_path = all_img_paths[row['image_id']]
            img = tf.keras.preprocessing.image.load_img(img_path, target_size=self.img_size)
            img = tf.keras.preprocessing.image.img_to_array(img)
            img = tf.keras.applications.efficientnet.preprocess_input(img)
            images.append(img)
            labels.append(row['label'])

        return np.array(images), np.array(labels)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

print("✅ Data Preprocessing done and DataGenerator ready.")


Metadata loaded: (10015, 7)
Valid dataset: (10015, 8)
Train samples: (8012, 8)
Validation samples: (2003, 8)
✅ Data Preprocessing done and DataGenerator ready.
